# 02 | Limpeza e modelagem

**Pergunta de negócio:** os registros são consistentes e íntegros o bastante para sustentar decisões sobre receita e rentabilidade?

As regras ficam centralizadas em `src/clean_data.py` para que notebook e pipeline usem o mesmo processo.

In [ ]:
from pathlib import Path; import sqlite3; import sys; import pandas as pd
root = Path.cwd().resolve(); assert (root / 'data' / 'raw').is_dir(), 'Execute primeiro o notebook 01'; sys.path.insert(0, str(root))

In [ ]:
raw = {name: pd.read_csv(root / 'data' / 'raw' / f'{name}.csv') for name in ['regions', 'customers', 'products', 'sellers', 'sales']}
pd.DataFrame([{'tabela': name, 'linhas': len(frame), 'nulos': int(frame.isna().sum().sum()), 'duplicatas': int(frame.duplicated().sum())} for name, frame in raw.items()])

## Regras aplicadas

A limpeza remove duplicatas por chave primária, padroniza espaços, valida datas, descarta vendas com quantidade/preço inválido ou desconto fora de 0–100%, e elimina chaves sem dimensão correspondente. O custo unitário vem do cadastro de produtos; a margem é **estimada**, não representa custo contábil real.

In [ ]:
from src.clean_data import clean_tables, load_sqlite
tables = clean_tables(); processed_dir = root / 'data' / 'processed'; processed_dir.mkdir(parents=True, exist_ok=True)
[frame.to_csv(processed_dir / f'{name}.csv', index=False, encoding='utf-8') for name, frame in tables.items()]
load_sqlite(tables); sales = tables['sales']; sales[['gross_revenue', 'discount_amount', 'net_revenue', 'estimated_cost', 'estimated_margin', 'margin_pct']].describe().round(2)

In [ ]:
database = processed_dir / 'retail_sales.db'; connection = sqlite3.connect(database); counts = pd.read_sql_query('SELECT (SELECT COUNT(*) FROM sales) AS itens, (SELECT COUNT(DISTINCT order_id) FROM sales) AS pedidos, (SELECT COUNT(DISTINCT customer_id) FROM sales) AS clientes_com_compra', connection); violations = pd.read_sql_query('PRAGMA foreign_key_check', connection); connection.close()
print(counts.to_string(index=False)); print(f'Violações de chave estrangeira: {len(violations)}'); assert violations.empty

In [ ]:
checks = {'IDs de venda únicos': sales.sale_id.is_unique, 'receita líquida não negativa': sales.net_revenue.ge(0).all(), 'desconto entre 0 e 100%': sales.discount_pct.between(0, 1).all(), 'margem calculada': sales.estimated_margin.notna().all()}; print(checks); assert all(checks.values())

## Resposta

Com unicidade nas vendas, chaves relacionais válidas e medidas financeiras derivadas em nível de item, a base pode alimentar consultas SQL e análises agregadas. O SQLite e os CSVs tratados ficam em `data/processed/`.